In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm


In [2]:
# --- 1. Define Paths to Artifacts ---
# These paths point to the input directories from your attached Kaggle Datasets.
MODEL_PATH = '/kaggle/input/finalmodel/final-deberta-v3-model'
MLB_PATH = '/kaggle/input/finalmodel/mlb.joblib'
CONFUSION_DICT_PATH = '/kaggle/input/confusion/confusion_dictionary.json'
TEST_DATA_PATH = '/kaggle/input/map-charting-student-math-misunderstandings/test.csv'
mod = '/kaggle/input/deberta-v3-base-offline-files/deberta-v3-base-offline'

In [3]:
tokenizer = AutoTokenizer.from_pretrained(mod)
model = AutoModelForSequenceClassification.from_pretrained(mod)

2025-09-12 15:01:54.291729: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757689314.623701      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757689314.717583      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
mlb = joblib.load(MLB_PATH)
labels = mlb.classes_

In [5]:
with open(CONFUSION_DICT_PATH, 'r') as f:
    confusion_dict = json.load(f)


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set model to evaluation mode

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


In [7]:
def predict_with_boost(text, row_id_debug=None):
    """
    Takes a single text string, gets the model's prediction,
    and applies a blended confusion dictionary boost to generate the top 3 labels.
    """
    # Tokenize the input text and move to the correct device
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get model's raw output (logits)
    with torch.no_grad():
        logits = model(**inputs).logits[0]

    # Get probabilities and the ranked order of predictions
    probabilities = torch.sigmoid(logits).cpu().numpy()
    ranked_indices = np.argsort(probabilities)[::-1]

    # =================================================================
    # =============== START: MODIFIED CODE BLOCK ======================
    # =================================================================
    # This entire block was rewritten to blend the model's predictions
    # with the confusion dictionary more effectively.

    model_top_labels = [labels[i] for i in ranked_indices[:3]]

    # CHANGED: Added a debug print to check the model's output for the first 5 rows
    if row_id_debug is not None and row_id_debug < 5:
        print(f"Debug (row {row_id_debug}): Model Top 1 is '{model_top_labels[0]}'")

    final_predictions = []
    
    # 1. Always take the model's best prediction
    top_1_label = model_top_labels[0]
    final_predictions.append(top_1_label)
    
    # 2. Get candidates from the confusion dictionary and the model's other top predictions
    confusion_candidates = confusion_dict.get(top_1_label, [])
    model_candidates = model_top_labels[1:]
    
    # 3. Interleave candidates to fill the remaining spots, prioritizing the dictionary's top choice
    
    # Add top confusion candidate if available and unique
    if len(confusion_candidates) > 0 and confusion_candidates[0] not in final_predictions:
        final_predictions.append(confusion_candidates[0])

    # Add model's 2nd choice if we still have space and it's unique
    if len(final_predictions) < 3 and len(model_candidates) > 0 and model_candidates[0] not in final_predictions:
        final_predictions.append(model_candidates[0])

    # 4. Fill any remaining spots with the rest of the unique candidates
    all_candidates = confusion_candidates + model_candidates
    for candidate in all_candidates:
        if len(final_predictions) >= 3:
            break
        if candidate not in final_predictions:
            final_predictions.append(candidate)
            
    # 5. Final fallback if we still don't have 3 predictions
    while len(final_predictions) < 3:
        final_predictions.append("False_Neither:NA") # A safe, common placeholder

    return final_predictions[:3]

In [8]:
print("Loading and preparing test data...")
test_df = pd.read_csv(TEST_DATA_PATH)


Loading and preparing test data...


In [9]:
test_df['input_text'] = test_df.apply(
    lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", 
    axis=1
)

print("Generating predictions for the test set...")
all_predictions = []
# Use tqdm for a progress bar
for text in tqdm(test_df['input_text'].tolist(), desc="Predicting"):
    top_3_labels = predict_with_boost(text)
    all_predictions.append(top_3_labels)


Generating predictions for the test set...


Predicting:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
print("Formatting and saving submission file...")

# The competition requires the ID and a space-separated string of the 3 labels
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'],
    'Category:Misconception': [' '.join(pred_list) for pred_list in all_predictions]
})

# Save to submission.csv
submission_df.to_csv('submission.csv', index=False)

print("="*50)
print("submission.csv created successfully!")
print("="*50)
print(submission_df.head())

Formatting and saving submission file...
submission.csv created successfully!
   row_id                             Category:Misconception
0   36696  False_Correct:nan False_Neither:nan False_Misc...
1   36697  False_Correct:nan False_Neither:nan False_Misc...
2   36698  False_Correct:nan False_Neither:nan False_Misc...
